In [17]:
# # Titanic Classification Pipeline
# Train model using pipelines, column transformers, grid search, and cross-validation.  
# Save results to `data/Output_Titanic.xlsx`.

In [18]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn import metrics
import warnings
warnings.filterwarnings("ignore")

In [19]:
# Load train data
train_file = 'data/train.csv'
train_df = pd.read_csv(train_file)

# Create 'Deck' from 'Cabin'
train_df['Deck'] = train_df['Cabin'].apply(lambda x: x[0] if pd.notna(x) else np.nan)

# Features and target
X = train_df.drop(['Survived'], axis=1)
y = train_df['Survived']

# Split into training and validation sets
Xtrain, Xval, ytrain, yval = train_test_split(X, y, test_size=0.2, random_state=1)

# Copy to avoid SettingWithCopyWarning
Xtrain = Xtrain.copy()
Xval = Xval.copy()
ytrain = ytrain.copy()
yval = yval.copy()

In [20]:
# Preprocessing pipelines
numeric_features = ['Age']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_features = ['Pclass', 'Sex', 'Deck']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='X')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', dtype=int))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')

In [21]:
# Pipeline with Logistic Regression
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(solver='liblinear'))
])

In [22]:
# GridSearchCV parameters
param_grid = {
    'lr__penalty': ['l1', 'l2'],
    'preprocessor__num__imputer__strategy': ['median', 'mean']
}

gscv = GridSearchCV(clf, param_grid, cv=5, return_train_score=False)
gscv.fit(Xtrain, ytrain)

best_estimator = gscv.best_estimator_

In [23]:
# Evaluate on validation data
ypred = best_estimator.predict(Xval)

accuracy = metrics.accuracy_score(yval, ypred)
conf_matrix = metrics.confusion_matrix(yval, ypred)
class_report = metrics.classification_report(yval, ypred, output_dict=True)

print("Validation Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", pd.DataFrame(class_report).T)

Validation Accuracy: 0.7821229050279329
Confusion Matrix:
 [[87 19]
 [20 53]]
Classification Report:
               precision    recall  f1-score     support
0              0.813084  0.820755  0.816901  106.000000
1              0.736111  0.726027  0.731034   73.000000
accuracy       0.782123  0.782123  0.782123    0.782123
macro avg      0.774598  0.773391  0.773968  179.000000
weighted avg   0.781693  0.782123  0.781883  179.000000


In [24]:
# Cross-validation scores on training data
cv_scores = cross_validate(best_estimator, Xtrain, ytrain, cv=5, return_train_score=False)
cv_mean = cv_scores['test_score'].mean()
print("Mean CV Score:", cv_mean)

Mean CV Score: 0.8020289569585344


In [25]:
# Save results to Excel
output_file = "data/Output_Titanic.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Accuracy
    pd.DataFrame({'Validation Accuracy':[accuracy]}).to_excel(writer, sheet_name='Accuracy', index=False)
    # Confusion matrix
    pd.DataFrame(conf_matrix).to_excel(writer, sheet_name='Confusion_Matrix', index=False)
    # Classification report
    pd.DataFrame(class_report).T.to_excel(writer, sheet_name='Classification_Report')
    # Cross-validation scores
    pd.DataFrame({'CV Score': cv_scores['test_score']}).to_excel(writer, sheet_name='CV_Scores', index=False)

print(f"Output saved to {output_file}")

Output saved to data/Output_Titanic.xlsx
